# SpaceX Falcon 9 First Stage Landing Prediction
## Data Collection: Web Scraping

In this notebook we collect Falcon 9 historical launch records by web scraping a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches` using the `BeautifulSoup` library, cross-checking the SpaceX REST API dataset collected earlier.

To keep results consistent and reproducible, we scrape a **fixed snapshot** of the Wikipedia page (a specific `oldid` revision) rather than the live, constantly-editable page.

## Objectives

* Request the Falcon 9 launch Wiki page from its URL
* Extract all column/variable names from the HTML table header
* Create a data frame by parsing the launch HTML tables

### Import libraries

In [ ]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd


### Helper functions

These functions extract each field of a table row from the raw HTML `<td>`/`<th>` cell objects.

In [ ]:
def date_time(table_cells):
    """
    Returns the data and time from the HTML table cell.
    Input: the element of a table data cell that extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    Returns the booster version from the HTML table cell.
    Input: the element of a table data cell that extracts extra row
    """
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """
    Returns the landing status from the HTML table cell.
    Input: the element of a table data cell that extracts extra row
    """
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """
    This function returns the column name from the HTML table header cell.
    Input: the element of a table header cell.
    """
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()

    column_name = ' '.join(row.contents)

    # Filter the digit and empty names
    if not column_name.strip().isdigit():
        column_name = column_name.strip()
        return column_name


### Request the Falcon 9 launch Wiki page from its URL

To keep the results of this notebook consistent, we scrape the Wikipedia page as it existed on a specific revision (`oldid`) rather than the live page, which changes over time.

In [ ]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"


Perform an HTTP GET request to fetch the Falcon 9 launch Wiki page as an HTTP response, then create a `BeautifulSoup` object from it.

In [ ]:
# Use requests.get() to request the Falcon 9 launch wiki page
response = requests.get(static_url)

# Create a BeautifulSoup object from the HTML response
soup = BeautifulSoup(response.text, 'html.parser')


In [ ]:
# Use soup.title to verify the page was loaded correctly
print(soup.title)


### Extract all column/variable names from the HTML table header

In [ ]:
# Find all tables on the wiki page first
html_tables = soup.find_all('table')
print(f"Found {len(html_tables)} tables on the page")


Starting from the third table is the target table containing the actual launch records.

In [ ]:
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(first_launch_table)


In [ ]:
column_names = []

# Apply find_all() function with 'th' element on first_launch_table
th_elements = first_launch_table.find_all('th')

# Iterate each th element and apply the extract_column_from_header() to get a column name
# Append the non-empty column name (name is not None and len(name) > 0) into a list called column_names
for th in th_elements:
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)


In [ ]:
print(column_names)


### Create a data frame by parsing the launch HTML tables

In [ ]:
launch_dict = dict.fromkeys(column_names)

# Remove irrelevant columns
del launch_dict['Date and time ( )']

# Initialize the launch_dict with each value as an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []

# Added some new columns
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []


Fill up the `launch_dict` by parsing each row of the launch tables. This mirrors the standard extraction logic: for every table on the page that has a `Launch No.`-style leading numeric column, pull out the flight number, date/time, booster version, launch site, payload, payload mass, orbit, customer, launch outcome, and booster landing outcome.

In [ ]:
extracted_row = 0

# Extract each table
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    # get table row
    for rows in table.find_all("tr"):
        # check to see if first table heading is as number corresponding to launch a number
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False

        # get table element
        row = rows.find_all('td')
        # if it is number save cells in a dictonary
        if flag:
            extracted_row += 1
            # Flight Number value
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])

            # Date value
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)

            # Time value
            time = datatimelist[1]
            launch_dict['Time'].append(time)

            # Booster version
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else None
            launch_dict['Version Booster'].append(bv)

            # Launch Site
            launch_site = row[2].a.string if row[2].a else row[2].get_text(strip=True)
            launch_dict['Launch site'].append(launch_site)

            # Payload
            payload = row[3].a.string if row[3].a else row[3].get_text(strip=True)
            launch_dict['Payload'].append(payload)

            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)

            # Orbit
            orbit = row[5].a.string if row[5].a else row[5].get_text(strip=True)
            launch_dict['Orbit'].append(orbit)

            # Customer
            customer = row[6].a.string if row[6].a else row[6].get_text(strip=True)
            launch_dict['Customer'].append(customer)

            # Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)

            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f"Extracted {extracted_row} launch rows")


Create a Pandas dataframe from `launch_dict`.

In [ ]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
df.head()


Export the scraped dataset to a CSV file, `spacex_web_scraped.csv`, for use as a cross-check against the SpaceX API dataset.

In [ ]:
df.to_csv('spacex_web_scraped.csv', index=False)


## Authors

Abhinav Jagtap